# Week 10: Does Teacher Architecture Matter in Knowledge Distillation?

## Research question

When a ViT student is trained with **distillation**, is the gain mainly from **CNN inductive bias** (DeiT-style CNN teacher), or from **soft-label regularization** that any calibrated teacher could provide?

## Four conditions (same student, optimizer, schedule)

| ID | Condition | Supervision |
|---|---|---|
| **C1** | Label smoothing | `CrossEntropyLoss(label_smoothing=0.1)` only — no teacher |
| **C2** | EMA self-distillation | Hard CE + KL vs. **EMA of the student's own weights** |
| **C3** | ViT teacher | Hard CE + KL vs. a **frozen ViT** trained separately (same arch) |
| **C4** | CNN teacher | Hard CE + KL vs. a **frozen ResNet-56** trained on CIFAR-10 |

## Loss (C2–C4)

$$
\mathcal{L} = (1-\alpha)\,\mathrm{CE}(\mathbf{z}_s, y) + \alpha\, T^2 \,\mathrm{KL}\big(\sigma(\mathbf{z}_s/T)\,\|\,\sigma(\mathbf{z}_t/T)\big)
$$

where $\mathbf{z}_s$ / $\mathbf{z}_t$ are student / teacher logits, $T$ is temperature, $\alpha$ mixes CE and KL.

## Primary metric

**Wall-clock time to reach 94% test accuracy** on CIFAR-10.

## Imports

In [ ]:
import sys
import math
import time
import copy
from pathlib import Path
from typing import Optional, Dict, Any, List, Tuple

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
DATA_DIR = PROJECT_ROOT / "data"

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import DataLoader

from src.model import count_parameters
from src.utils import (
    get_device,
    get_cifar10_loaders,
    get_model,
    set_seed,
    validate,
    get_peak_gpu_memory_mb,
    reset_peak_gpu_memory,
)

print(f"PyTorch: {torch.__version__}")
device = get_device()
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"BF16: {torch.cuda.is_bf16_supported()}")

## Config

In [ ]:
SEED = 42
TARGET_ACCURACY = 94.0
MAX_EPOCHS = 100
BATCH_SIZE = 512
NUM_WORKERS = 4
LR = 1e-3
WARMUP_EPOCHS = 5
TEACHER_EPOCHS = 60
DISTILL_ALPHA = 0.5
DISTILL_TEMP = 3.0
EMA_DECAY = 0.999

set_seed(SEED, deterministic=False)
print(f"Target {TARGET_ACCURACY}% | α={DISTILL_ALPHA} T={DISTILL_TEMP} | EMA decay={EMA_DECAY}")

## Data

In [ ]:
train_loader, test_loader = get_cifar10_loaders(
    data_dir=str(DATA_DIR),
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    augment_train=True,
    use_randaugment=True,
    randaugment_num_ops=2,
    randaugment_magnitude=9,
    persistent_workers=NUM_WORKERS > 0,
    prefetch_factor=2,
)
xb, _ = next(iter(train_loader))
print(f"Batches: {len(train_loader)} | sample batch: {xb.shape}")

## Student (ViT, same as `get_model` in `src.utils`)

In [ ]:
def make_student() -> nn.Module:
    return get_model(patch_size=4, embed_dim=192, depth=6, num_heads=6, dropout=0.0)

_s = make_student().to(device)
print(f"Student params: {count_parameters(_s):,}")
print(f"Output: {_s(xb.to(device)).shape}")
del _s

## CNN teacher: ResNet-56 for CIFAR-10

In [ ]:
class BasicBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, stride: int = 1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch),
            )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + self.shortcut(x))


class ResNetCIFAR(nn.Module):
    """ResNet-56: n=9 blocks per stage."""
    def __init__(self, n: int = 9, num_classes: int = 10):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
        )
        self.layer1 = self._make_layer(16, 16, n, 1)
        self.layer2 = self._make_layer(16, 32, n, 2)
        self.layer3 = self._make_layer(32, 64, n, 2)
        self.head = nn.Linear(64, num_classes)
        self._init_weights()

    def _make_layer(self, in_ch, out_ch, n, stride):
        layers = [BasicBlock(in_ch, out_ch, stride)]
        for _ in range(n - 1):
            layers.append(BasicBlock(out_ch, out_ch))
        return nn.Sequential(*layers)

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = x.mean(dim=(2, 3))
        return self.head(x)

print("ResNetCIFAR defined.")

## Train frozen teachers (CNN + ViT)

In [ ]:
def train_teacher(
    model: nn.Module,
    label: str,
    epochs: int = TEACHER_EPOCHS,
    lr: float = 1e-3,
) -> nn.Module:
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.05)
    crit = nn.CrossEntropyLoss(label_smoothing=0.1)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=lr * 0.01)
    use_amp = device.type == "cuda"
    dtype = torch.bfloat16 if use_amp and torch.cuda.is_bf16_supported() else torch.float16
    best, best_sd = 0.0, None
    print(f"Training {label} ({epochs} epochs)...")
    for ep in range(1, epochs + 1):
        model.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad(set_to_none=True)
            if use_amp:
                with torch.amp.autocast(device_type="cuda", dtype=dtype):
                    loss = crit(model(x), y)
                loss.backward()
            else:
                loss = crit(model(x), y)
                loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        sched.step()
        _, acc = validate(model, test_loader, nn.CrossEntropyLoss(), device)
        if acc > best:
            best, best_sd = acc, copy.deepcopy(model.state_dict())
        if ep % 10 == 0 or ep == epochs:
            print(f"  [{label}] ep {ep}/{epochs} test {acc:.2f}% best {best:.2f}%")
    model.load_state_dict(best_sd)
    model.eval()
    for p in model.parameters():
        p.requires_grad_(False)
    print(f"{label} done. Best test: {best:.2f}%\n")
    return model


set_seed(SEED)
cnn_teacher = train_teacher(ResNetCIFAR(n=9), "CNN (ResNet-56)", lr=1e-3)
set_seed(SEED + 1)
vit_teacher = train_teacher(make_student(), "ViT (same arch)", lr=LR)

## Distillation utilities

In [ ]:
def distill_loss(student_logits, teacher_logits, y, alpha=DISTILL_ALPHA, T=DISTILL_TEMP):
    ce = F.cross_entropy(student_logits, y)
    kl = F.kl_div(
        F.log_softmax(student_logits / T, dim=-1),
        F.softmax(teacher_logits.detach() / T, dim=-1),
        reduction="batchmean",
    ) * (T ** 2)
    return (1 - alpha) * ce + alpha * kl


class EMATeacher:
    def __init__(self, student: nn.Module, decay: float = EMA_DECAY):
        self.decay = decay
        self.model = copy.deepcopy(student)
        self.model.eval()
        for p in self.model.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, student: nn.Module):
        for ep, sp in zip(self.model.parameters(), student.parameters()):
            ep.data.mul_(self.decay).add_(sp.data, alpha=1.0 - self.decay)

    @torch.no_grad()
    def logits(self, x):
        return self.model(x)

print("distill_loss + EMATeacher ready.")

## Unified student training (`train_distill`)

In [ ]:
def train_distill(
    student: nn.Module,
    condition: str,
    teacher: Optional[nn.Module] = None,
    num_epochs: int = MAX_EPOCHS,
) -> Dict[str, Any]:
    assert condition in ("label_smooth", "ema", "vit_teacher", "cnn_teacher")
    if condition in ("vit_teacher", "cnn_teacher") and teacher is None:
        raise ValueError("external teacher required")

    student = student.to(device)
    # Note: avoid torch.compile here — EMA uses deepcopy(student) and compiled modules can break that.
    ema = EMATeacher(student) if condition == "ema" else None
    if teacher is not None:
        teacher = teacher.to(device)

    ls = 0.1 if condition == "label_smooth" else 0.0
    ce_only = nn.CrossEntropyLoss(label_smoothing=ls)

    opt = torch.optim.AdamW(student.parameters(), lr=LR, weight_decay=0.01)

    def lr_lambda(ep):
        if ep < WARMUP_EPOCHS:
            return (ep + 1) / WARMUP_EPOCHS
        p = (ep - WARMUP_EPOCHS) / max(1, num_epochs - WARMUP_EPOCHS)
        return 0.5 * (1 + math.cos(math.pi * p))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)

    use_amp = device.type == "cuda"
    dtype = torch.bfloat16 if use_amp and torch.cuda.is_bf16_supported() else torch.float16

    hist = {"test_acc": [], "train_loss": [], "wall_time": []}
    ttt, best = None, 0.0
    t0 = time.perf_counter()

    for ep in range(1, num_epochs + 1):
        student.train()
        tot_loss = 0.0
        t_ep = time.perf_counter()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad(set_to_none=True)
            if use_amp:
                with torch.amp.autocast(device_type="cuda", dtype=dtype):
                    z_s = student(x)
                    if condition == "label_smooth":
                        loss = ce_only(z_s, y)
                    else:
                        if condition == "ema":
                            z_t = ema.logits(x)
                        else:
                            with torch.no_grad():
                                z_t = teacher(x)
                        loss = distill_loss(z_s, z_t, y)
            else:
                z_s = student(x)
                if condition == "label_smooth":
                    loss = ce_only(z_s, y)
                else:
                    if condition == "ema":
                        z_t = ema.logits(x)
                    else:
                        with torch.no_grad():
                            z_t = teacher(x)
                    loss = distill_loss(z_s, z_t, y)
            loss.backward()
            nn.utils.clip_grad_norm_(student.parameters(), 1.0)
            opt.step()
            if ema is not None:
                ema.update(student)
            tot_loss += loss.item()
        sched.step()
        wt = time.perf_counter() - t_ep
        _, acc = validate(student, test_loader, nn.CrossEntropyLoss(), device)
        best = max(best, acc)
        if ttt is None and acc >= TARGET_ACCURACY:
            ttt = time.perf_counter() - t0
        hist["test_acc"].append(acc)
        hist["train_loss"].append(tot_loss / len(train_loader))
        hist["wall_time"].append(wt)
        tag = f" [→{TARGET_ACCURACY}% @ {ttt:.0f}s]" if ttt else ""
        print(f"[{condition:14s}] ep{ep:3d} loss {hist['train_loss'][-1]:.4f} test {acc:.2f}%{tag}")
        if ttt is not None:
            break

    return {
        "condition": condition,
        "history": hist,
        "best_acc": best,
        "time_to_target": ttt,
        "total_time": time.perf_counter() - t0,
        "peak_gpu_mb": get_peak_gpu_memory_mb(),
    }

print("train_distill defined.")

## Run all four conditions

In [ ]:
all_results: Dict[str, Dict[str, Any]] = {}
runs = [
    ("label_smooth", None, "C1: label smoothing"),
    ("ema", None, "C2: EMA self-distill"),
    ("vit_teacher", vit_teacher, "C3: ViT teacher"),
    ("cnn_teacher", cnn_teacher, "C4: CNN teacher"),
]
for cond, teach, name in runs:
    print("=" * 56)
    print(name)
    set_seed(SEED)
    reset_peak_gpu_memory()
    all_results[name] = train_distill(make_student(), cond, teach)
    r = all_results[name]
    print(f"  best={r['best_acc']:.2f}%  ttt={r['time_to_target']}\n")

## Summary table

In [ ]:
c1 = all_results.get("C1: label smoothing", {}).get("time_to_target")
print(f"{'Condition':<22} {'ttt 94%':>12} {'best%':>8}  vs C1")
print("-" * 52)
for name, r in all_results.items():
    t = r["time_to_target"]
    ts = f"{t:.1f}s" if t else "—"
    sp = f"{c1/t:.2f}x" if (c1 and t) else ("—" if not t else "base")
    print(f"{name:<22} {ts:>12} {r['best_acc']:>7.2f}%  {sp}")

## Plots

In [ ]:
import matplotlib.pyplot as plt

colors = {
    "C1: label smoothing": "#6c757d",
    "C2: EMA self-distill": "#0077b6",
    "C3: ViT teacher": "#f77f00",
    "C4: CNN teacher": "#d62828",
}
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
fig.suptitle("Teacher architecture ablation", fontweight="bold")

for name, r in all_results.items():
    axes[0].plot(r["history"]["test_acc"], label=name, color=colors[name])
axes[0].axhline(TARGET_ACCURACY, color="k", ls="--", lw=0.8, alpha=0.5)
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("test acc %")
axes[0].legend(fontsize=7)
axes[0].grid(alpha=0.3)

for name, r in all_results.items():
    w = r["history"]["wall_time"]
    cum = [sum(w[: i + 1]) for i in range(len(w))]
    axes[1].plot(cum, r["history"]["test_acc"], label=name, color=colors[name])
axes[1].axhline(TARGET_ACCURACY, color="k", ls="--", lw=0.8, alpha=0.5)
axes[1].set_xlabel("wall time (s)")
axes[1].set_ylabel("test acc %")
axes[1].legend(fontsize=7)
axes[1].grid(alpha=0.3)

names_b = [n for n, r in all_results.items() if r["time_to_target"] is not None]
vals = [all_results[n]["time_to_target"] for n in names_b]
if vals:
    axes[2].bar(range(len(names_b)), vals, color=[colors[n] for n in names_b])
    axes[2].set_xticks(range(len(names_b)))
    axes[2].set_xticklabels(names_b, rotation=15, ha="right", fontsize=7)
    axes[2].set_ylabel("s to 94%")
else:
    axes[2].text(0.5, 0.5, "No run hit 94%", ha="center", va="center", transform=axes[2].transAxes)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "notebooks" / "week10_results.png", dpi=150, bbox_inches="tight")
plt.show()

## How to read results

- **C4 ≫ others** → CNN teacher’s inductive bias seems to matter for this student/dataset.
- **C2 ≈ C3 ≈ C4 ≫ C1** → soft supervision matters; teacher *architecture* may not.
- **C1 ≈ all** → gains collapse to label-style regularization; distillation may not add much beyond C1 here.
- **C2 ≈ C4** → EMA self-distillation can match an external CNN teacher (interesting for cost).